# Similarity Search and Ranking Metrics

AI & ML Foundations: Embeddings and retrieval · *Mathematics for AI and Machine Learning* by Fuheng Wu

Rank items by cosine similarity to a query and score the ranking with P@K, R@K and Average Precision (Chapters 1 and 14).

Companion code for the [interactive demo](https://math4ai.org/companion/ml/similarity-ranking/) on math4ai.org (sign-in required). Run the cell with Shift+Enter; PyTorch is preinstalled on Colab. Change the values at the top and run it again to explore.


In [1]:
import math

import torch
import torch.nn.functional as F

torch.manual_seed(9)


def precision_at_k(rel, k):
    return rel[:k].float().mean().item()


def recall_at_k(rel, k):
    return (rel[:k].sum() / rel.sum()).item()


def average_precision(rel):
    """AP = mean of P@k over the positions k where a relevant item appears."""
    hits = rel.float().cumsum(0)
    ranks = torch.arange(1, len(rel) + 1)
    return ((hits / ranks) * rel).sum().item() / rel.sum().item()


# ── The book's worked example (Chapter 14): MAP of two queries ──────────────
q1 = torch.tensor([1, 0, 1, 0, 1])
q2 = torch.tensor([1, 1, 0, 1])
ap1, ap2 = average_precision(q1), average_precision(q2)
print(f"AP1 = {ap1:.3f}, AP2 = {ap2:.3f}, MAP = {(ap1 + ap2) / 2:.4f}   (book: 0.756, 0.917, 0.837 from the rounded APs)")

# ── The demo's setup: 30 embeddings in 3 topics, rank by cosine to a query ──
centers = torch.tensor([30.0, 150.0, 270.0]) * math.pi / 180
spread, query_angle, K = 0.35, math.radians(40), 5
topic = torch.arange(3).repeat_interleave(10)
angles = centers[topic] + spread * torch.randn(30)
lengths = 1 + 0.3 * torch.randn(30)                       # different lengths: cosine ignores them
X = torch.stack([lengths * angles.cos(), lengths * angles.sin()], 1)
q = torch.tensor([math.cos(query_angle), math.sin(query_angle)])

cos = F.cosine_similarity(X, q[None], dim=1)
order = cos.argsort(descending=True)
q_topic = torch.cos(query_angle - centers).argmax()
rel = (topic[order] == q_topic).long()
print(f"\nquery at 40 deg -> relevant topic {int(q_topic)}; top {K} relevance: {rel[:K].tolist()}")
print(f"P@{K} = {precision_at_k(rel, K):.3f}, R@{K} = {recall_at_k(rel, K):.3f}, AP = {average_precision(rel):.3f}")

# Cosine vs raw dot product: the dot product also rewards long vectors
dot_order = (X @ q).argsort(descending=True)
print("same top-5 with the raw dot product?", order[:5].tolist() == dot_order[:5].tolist())

# ── Wider topics overlap, and the ranking gets worse (mean AP over 50 draws) ─
print("\n spread   mean AP")
for s in (0.1, 0.35, 0.7, 1.2):
    aps = []
    for _ in range(50):
        a = centers[topic] + s * torch.randn(30)
        Xs = torch.stack([a.cos(), a.sin()], 1)
        r = (topic[F.cosine_similarity(Xs, q[None], dim=1).argsort(descending=True)] == q_topic).long()
        aps.append(average_precision(r))
    print(f"{s:6.2f}   {sum(aps) / len(aps):.3f}")

AP1 = 0.756, AP2 = 0.917, MAP = 0.8361   (book: 0.756, 0.917, 0.837 from the rounded APs)

query at 40 deg -> relevant topic 0; top 5 relevance: [1, 1, 1, 1, 1]
P@5 = 1.000, R@5 = 0.500, AP = 1.000
same top-5 with the raw dot product? False

 spread   mean AP
  0.10   1.000
  0.35   1.000
  0.70   0.940
  1.20   0.654
